# 05 - Cannibalization Analysis
## Detect product substitution and calculate cannibalization metrics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (14, 6)
%matplotlib inline

In [ ]:
DATA_DIR = '../data/raw/'

transactions = pd.read_csv(f'{DATA_DIR}transaction_data.csv')
products = pd.read_csv(f'{DATA_DIR}product.csv')
campaign_desc = pd.read_csv(f'{DATA_DIR}campaign_desc.csv')
campaign_table = pd.read_csv(f'{DATA_DIR}campaign_table.csv')

df = transactions.merge(products[['PRODUCT_ID', 'DEPARTMENT', 'COMMODITY_DESC']], on='PRODUCT_ID', how='left')
print(f'Merged dataset: {len(df):,} rows')

In [ ]:
def detect_cannibalization(campaign_id):
    camp = campaign_desc[campaign_desc['CAMPAIGN'] == campaign_id]
    if camp.empty:
        return None
    
    start = int(camp['START_DAY'].iloc[0])
    end = int(camp['END_DAY'].iloc[0])
    
    pre = df[(df['DAY'] >= start - 28) & (df['DAY'] < start)]
    during = df[(df['DAY'] >= start) & (df['DAY'] <= end)]
    
    pre_product = pre.groupby('PRODUCT_ID', as_index=False).agg(pre_qty=('QUANTITY', 'sum'))
    during_product = during.groupby('PRODUCT_ID', as_index=False).agg(during_qty=('QUANTITY', 'sum'))
    
    merged = pre_product.merge(during_product, on='PRODUCT_ID', how='outer').fillna(0)
    
    pre_days = max(1, pre['DAY'].nunique())
    during_days = max(1, end - start + 1)
    
    merged['expected_qty'] = (merged['pre_qty'] / pre_days) * during_days
    merged['change'] = merged['during_qty'] - merged['expected_qty']
    
    top_during = during.groupby('PRODUCT_ID')['QUANTITY'].sum().nlargest(5).index
    
    affected = merged[
        (merged['change'] < -1) &
        (~merged['PRODUCT_ID'].isin(top_during))
    ].sort_values('change')
    
    affected_products = []
    total_lost = 0.0
    for _, ap in affected.head(10).iterrows():
        lost = abs(float(ap['change']))
        total_lost += lost
        pinfo = products[products['PRODUCT_ID'] == ap['PRODUCT_ID']]
        pname = str(pinfo['COMMODITY_DESC'].values[0]) if not pinfo.empty else 'Unknown'
        affected_products.append({
            'product_id': int(ap['PRODUCT_ID']),
            'product_name': pname,
            'estimated_lost_sales': round(lost, 2),
        })
    
    promoted_id = int(top_during[0]) if len(top_during) > 0 else None
    during_qty = during['QUANTITY'].sum()
    cannibal_score = round(total_lost / during_qty, 4) if during_qty > 0 else 0
    
    return {
        'campaign_id': campaign_id,
        'promoted_product_id': promoted_id,
        'affected_products': affected_products,
        'total_lost_sales': round(total_lost, 2),
        'cannibalization_score': min(cannibal_score, 1.0),
    }

In [ ]:
# Test on one campaign
result = detect_cannibalization(1)
result

In [ ]:
# Analyze all campaigns
all_results = []
for cid in campaign_desc['CAMPAIGN'].unique():
    r = detect_cannibalization(cid)
    if r:
        all_results.append(r)

cannibal_df = pd.DataFrame(all_results)
cannibal_df = cannibal_df.sort_values('cannibalization_score', ascending=False)
print(f'Analyzed {len(cannibal_df)} campaigns')
cannibal_df[['campaign_id', 'cannibalization_score', 'total_lost_sales', 'promoted_product_id']].head(10)

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
plt.hist(cannibal_df['cannibalization_score'], bins=20, alpha=0.7, edgecolor='black')
plt.title('Distribution of Cannibalization Scores')
plt.xlabel('Cannibalization Score')
plt.ylabel('Number of Campaigns')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(range(len(cannibal_df)), cannibal_df['cannibalization_score'], alpha=0.6, s=60)
plt.axhline(y=0.2, color='red', linestyle='--', label='High cannibalization threshold')
plt.title('Cannibalization Score by Campaign')
plt.xlabel('Campaign Index')
plt.ylabel('Score')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary
print('=== Cannibalization Summary ===')
high_cannibal = cannibal_df[cannibal_df['cannibalization_score'] > 0.2]
print(f'Campaigns with high cannibalization (>0.2): {len(high_cannibal)} / {len(cannibal_df)}')
print(f'\nCannibalization score stats:')
print(cannibal_df['cannibalization_score'].describe())
print(f'\nAverage lost sales per campaign: ${cannibal_df["total_lost_sales"].mean():.2f}')